In [ ]:
import pygame
import sys

# ==================== CLASE PERSONAJE ====================
class Personaje:
    def __init__(self, clase, agilidad, vitalidad):
        self.clase = clase
        self.agilidad = agilidad
        self.vitalidad = vitalidad
    def __repr__(self):
        return f"Personaje(clase={self.clase}, agilidad={self.agilidad}, vitalidad={self.vitalidad})"


def main():
    # ==================== INICIALIZACIÓN DE PYGAME ====================
    pygame.init()

    # ==================== CONFIGURACIÓN DE LA VENTANA ====================
    ANCHO, ALTO = 800, 600
    pantalla = pygame.display.set_mode((ANCHO, ALTO))
    pygame.display.set_caption("Menú principal - Grupo 5")

    # ==================== DEFINICIÓN DE COLORES ====================
    BLANCO = (255, 255, 255)
    NEGRO = (0, 0, 0)
    AZUL = (0, 100, 255)
    AZUL_CLARO = (100, 150, 255)

    # ==================== DEFINICIÓN DE FUENTES ====================
    fuente_titulo = pygame.font.Font(None, 72)
    fuente_boton = pygame.font.Font(None, 48)
    fuente_texto = pygame.font.Font(None, 36)

    # ==================== CONFIGURACIÓN DE RELOJ Y FPS ====================
    reloj = pygame.time.Clock()
    FPS = 60

    # ==================== CLASE BOTÓN ====================
    class Boton:
        def __init__(self, x, y, ancho, alto, texto):
            self.rect = pygame.Rect(x, y, ancho, alto)
            self.texto = texto
            self.color_base = AZUL
            self.color_hover = AZUL_CLARO
            self.color_actual = self.color_base

        def dibujar(self, superficie):
            pygame.draw.rect(superficie, self.color_actual, self.rect)
            pygame.draw.rect(superficie, BLANCO, self.rect, 3)
            texto_surface = fuente_boton.render(self.texto, True, BLANCO)
            texto_rect = texto_surface.get_rect(center=self.rect.center)
            superficie.blit(texto_surface, texto_rect)

        def actualizar(self, pos_mouse):
            if self.rect.collidepoint(pos_mouse):
                self.color_actual = self.color_hover
            else:
                self.color_actual = self.color_base

        def fue_clickeado(self, pos_mouse):
            return self.rect.collidepoint(pos_mouse)

    # ==================== CREACIÓN DE BOTONES DEL MENÚ PRINCIPAL ====================
    boton_crear = Boton(ANCHO // 2 - 150, ALTO // 2 - 50, 300, 80, "Crear personaje")
    boton_salir = Boton(ANCHO // 2 - 150, ALTO // 2 + 100, 300, 80, "Salir del juego")

    # ==================== CREACIÓN DE BOTONES DE SELECCIÓN DE CLASES ====================
    btn_w, btn_h = 300, 60
    start_y = ALTO // 2 - 150
    clases = [("Clérigo", 'clerigo'), ("Guerrero", 'guerrero'), ("Mago", 'mago'), ("Luchador", 'luchador')]
    botones_clases = []
    for i, (label, key) in enumerate(clases):
        y = start_y + i * (btn_h + 10)
        botones_clases.append((key, Boton(ANCHO // 2 - btn_w // 2, y, btn_w, btn_h, label)))
    boton_volver = Boton(ANCHO // 2 - btn_w // 2, start_y + len(clases) * (btn_h + 10) + 20, btn_w, btn_h, "Volver atrás")
    
    # ==================== CREACIÓN DE BOTONES DE LA PANTALLA DE RESUMEN ====================
    boton_resumen_volver = Boton(ANCHO // 2 - 150, ALTO - 140, 300, 60, "Volver atrás")

    # ==================== VARIABLES DE CONTROL DE ESTADO ====================
    ejecutando = True
    en_menu = True
    en_crear = False
    en_resumen = False
    personaje_creado = None

    # ==================== ESTADÍSTICAS BASE POR CLASE ====================
    stats = {'clerigo': (5, 12), 'guerrero': (7, 15), 'mago': (8, 8), 'luchador': (10, 10)}

    # ==================== BUCLE PRINCIPAL ====================
    while ejecutando:
        reloj.tick(FPS)
        pos_mouse = pygame.mouse.get_pos()

        # ==================== MANEJO DE EVENTOS ====================
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                ejecutando = False
            if evento.type == pygame.MOUSEBUTTONDOWN and evento.button == 1:
                # --- Eventos en menú principal ---
                if en_menu:
                    if boton_crear.fue_clickeado(pos_mouse):
                        en_menu = False
                        en_crear = True
                    if boton_salir.fue_clickeado(pos_mouse):
                        ejecutando = False
                # --- Eventos en selección de clases ---
                elif en_crear and not en_resumen:
                    for key, btn in botones_clases:
                        if btn.fue_clickeado(pos_mouse):
                            agi, vit = stats[key]
                            personaje_creado = Personaje(key.capitalize(), agi, vit)
                            en_resumen = True
                            en_crear = False
                            break
                    if boton_volver.fue_clickeado(pos_mouse) and not en_resumen:
                        en_crear = False
                        en_menu = True
                # --- Eventos en pantalla de resumen ---
                elif en_resumen and personaje_creado is not None:
                    if boton_resumen_volver.fue_clickeado(pos_mouse):
                        personaje_creado = None
                        en_resumen = False
                        en_crear = True

        # ==================== ACTUALIZACIÓN DE BOTONES (HOVER) ====================
        if en_menu:
            boton_crear.actualizar(pos_mouse)
            boton_salir.actualizar(pos_mouse)
        elif en_crear and not en_resumen:
            for _, btn in botones_clases:
                btn.actualizar(pos_mouse)
            boton_volver.actualizar(pos_mouse)
        elif en_resumen and personaje_creado is not None:
            boton_resumen_volver.actualizar(pos_mouse)

        # ==================== DIBUJO DE PANTALLA ====================
        pantalla.fill(NEGRO)
        # --- Pantalla de menú principal ---
        if en_menu:
            titulo = fuente_titulo.render("Menú Principal", True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 100))
            pantalla.blit(titulo, titulo_rect)
            boton_crear.dibujar(pantalla)
            boton_salir.dibujar(pantalla)
        # --- Pantalla de selección de clases ---
        elif en_crear and not en_resumen:
            titulo = fuente_titulo.render("Seleccione una clase", True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 80))
            pantalla.blit(titulo, titulo_rect)
            for _, btn in botones_clases:
                btn.dibujar(pantalla)
            boton_volver.dibujar(pantalla)
        # --- Pantalla de resumen ---
        elif en_resumen and personaje_creado is not None:
            titulo = fuente_titulo.render("Personaje Creado", True, BLANCO)
            pantalla.blit(titulo, titulo.get_rect(center=(ANCHO // 2, 80)))
            lines = [f"Clase: {personaje_creado.clase}", f"Agilidad: {personaje_creado.agilidad}", f"Vitalidad: {personaje_creado.vitalidad}"]
            for i, line in enumerate(lines):
                surf = fuente_texto.render(line, True, BLANCO)
                rect = surf.get_rect(center=(ANCHO // 2, 180 + i * 40))
                pantalla.blit(surf, rect)
            nota = fuente_texto.render("No hay más interacción. Cierra la ventana para salir.", True, AZUL_CLARO)
            pantalla.blit(nota, nota.get_rect(center=(ANCHO // 2, ALTO - 80)))
            boton_resumen_volver.dibujar(pantalla)

        # ==================== ACTUALIZACIÓN DE PANTALLA ====================
        pygame.display.flip()

    # ==================== CIERRE DE PYGAME ====================
    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()